# California Single-Family Residence — Close Price Prediction
## 05 · Feature Engineering: Geographic & Temporal Features

`model_comparison.ipynb` established Decision Tree as the model to beat, with
Linear Regression (Ridge) as the interpretable baseline. The question explored in this notebook: **do these
*specific* engineered features help, on validation, checked with the same
bootstrap rigor used for model selection** 

**Four engineered features**

1. **Bed/bath ratio & property age at sale**: simple derived ratios, cheap
   to compute, standard in real-estate modeling.
2. **Cyclical month encoding** (`sin`/`cos` of close month): motivated by the
   seasonal pattern found in `data_exploration.ipynb`'s monthly price trend.
3. **School district via real spatial join** against CA School District Areas
   2024-25 boundaries — a second, independently-derived source for
   school-zone information, motivated by `HighSchoolDistrict`'s ~26%
   missingness found in exploration.
4. **ZIP-level historical price-per-sqft**: a locational aggregate feature,
   computed with **CV-safe target encoding** (K-fold out-of-fold assignment
   for training rows) so no row's own price leaks into its own feature


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("outputs")
RIDGE_ALPHA = 100.0  # from the validation alpha sweep in model_comparison.ipynb — update if yours differed


## Load the Primary Split

This notebook focuses on the primary split rather than re-running the full
rolling-origin backtest because the spatial join is comparatively expensive to
recompute per cutoff


In [2]:
with open(OUTPUT_DIR / "preprocessing_metadata.json", "r") as f:
    metadata = json.load(f)

TARGET_COL = metadata["target_col"]
PRIMARY_SPLIT = metadata["primary_split"]

base_numeric_features = metadata["numeric_features"] + metadata["missing_flag_cols"]
base_categorical_features = metadata["categorical_features"] + metadata["boolean_features"]
dtype_hint = {col: str for col in base_categorical_features}

suffix = PRIMARY_SPLIT.replace("-", "")
train_df = pd.read_csv(OUTPUT_DIR / f"train_{suffix}_split.csv", low_memory=False, dtype=dtype_hint)
val_df = pd.read_csv(OUTPUT_DIR / f"val_{suffix}_split.csv", low_memory=False, dtype=dtype_hint)
test_df = pd.read_csv(OUTPUT_DIR / f"test_{suffix}_split.csv", low_memory=False, dtype=dtype_hint)

old_numeric_features = [c for c in base_numeric_features if c in train_df.columns]
old_categorical_features = [c for c in base_categorical_features if c in train_df.columns]

print(f"Primary split: {PRIMARY_SPLIT}")
print(f"Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}")


Primary split: 2026-06
Train: (382529, 42)  Val: (11887, 42)  Test: (12696, 42)


## Feature 1-2: Bed/Bath Ratio & Property Age at Sale

`property_age_years` uses the sale year, not the current year, so the feature
reflects the property's age *at the time of that historical sale* rather than
its age today.

In [3]:
def engineer_bed_bath_age(df):
    df = df.copy()
    df["bed_bath_ratio"] = df["BedroomsTotal"] / df["BathroomsTotalInteger"].replace(0, np.nan)

    close_date = pd.to_datetime(df["CloseDate"], errors="coerce")
    df["property_age_years"] = close_date.dt.year - df["YearBuilt"]
    df.loc[df["property_age_years"] < 0, "property_age_years"] = np.nan
    return df

train_df = engineer_bed_bath_age(train_df)
val_df = engineer_bed_bath_age(val_df)
test_df = engineer_bed_bath_age(test_df)

train_df[["bed_bath_ratio", "property_age_years"]].describe()


,bed_bath_ratio,property_age_years
count,382262.000000,382251.000000
mean,1.465386,48.949154
std,0.475945,27.343718
min,0.011429,0.000000
25%,1.000000,27.000000
50%,1.500000,49.000000
75%,1.500000,69.000000
max,11.000000,249.000000


## Feature 3: Cyclical Month Encoding

`data_exploration.ipynb` found a repeating annual seasonal cycle in median
price (spring/summer peak, winter trough). A raw month number (1-12) would
force the model to treat December and January as maximally far apart, which
is wrong. `sin`/`cos` encoding maps month onto a circle so
the model sees December and January as neighbors, the way they actually are.


In [4]:
def engineer_cyclical_month(df):
    df = df.copy()
    month = pd.to_datetime(df["CloseDate"], errors="coerce").dt.month
    df["close_month_sin"] = np.sin(2 * np.pi * month / 12)
    df["close_month_cos"] = np.cos(2 * np.pi * month / 12)
    return df

train_df = engineer_cyclical_month(train_df)
val_df = engineer_cyclical_month(val_df)
test_df = engineer_cyclical_month(test_df)

train_df[["close_month_sin", "close_month_cos"]].describe()


,close_month_sin,close_month_cos
count,3.825290e+05,3.825290e+05
mean,-2.579043e-02,-3.389800e-02
std,7.177755e-01,6.949725e-01
min,-1.000000e+00,-1.000000e+00
25%,-8.660254e-01,-5.000000e-01
50%,-2.449294e-16,-1.836970e-16
75%,8.660254e-01,5.000000e-01
max,1.000000e+00,1.000000e+00


## Feature 4: School District via Real Spatial Join

`HighSchoolDistrict` (self-reported by the listing) was missing for ~26% of
rows in exploration. A spatial join against the official CA School District
Areas 2024-25 boundaries gives a independently-derived source for the
same information: every row with valid coordinates gets a district, whether
or not the MLS field was populated.


In [7]:
DISTRICTS_PATH = Path("outputs/California_School_District_Areas_2024-25.geojson")

if not DISTRICTS_PATH.exists():
    print(f"Looking for school district boundaries at: {DISTRICTS_PATH.resolve()}")
    resources_dir = Path("resources")
    if resources_dir.exists():
        print("Contents of resources/:", [p.name for p in resources_dir.iterdir()])
    else:
        print("resources/ directory not found either.")
    raise FileNotFoundError(
        f"{DISTRICTS_PATH} not found. Download the CA School District Areas 2024-25 "
        "boundaries from https://data.ca.gov/dataset/california-school-district-areas-2024-25 "
        "and update DISTRICTS_PATH above to match the file you saved (.shp/.geojson/.gpkg all work)."
    )

districts_gdf = gpd.read_file(DISTRICTS_PATH)
if districts_gdf.crs is None or districts_gdf.crs.to_epsg() != 4326:
    districts_gdf = districts_gdf.to_crs(epsg=4326)

print("Loaded district boundaries:", districts_gdf.shape)
districts_gdf.head()


Loaded district boundaries: (937, 54)


,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictName,DistrictType,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAreaSqMi,Shape__Area,Shape__Length,geometry
0,1,2024-25,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1356,12.6,3958,36.7,11.248939,4.755489e+07,56522.982683,"MULTIPOLYGON (((-122.22678 37.72651, -122.2267..."
1,2,2024-25,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,357,9.7,1184,32.1,1.789984,7.096327e+06,12696.382797,"POLYGON ((-122.28671 37.89852, -122.28673 37.8..."
2,3,2024-25,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1111,12.1,2686,29.3,10.434329,4.364648e+07,43695.341538,"POLYGON ((-122.25606 37.89834, -122.25607 37.8..."
3,4,2024-25,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,0,0.0,1113,11.6,3728,39.0,66.885571,2.838285e+08,142492.767565,"MULTIPOLYGON (((-122.01375 37.64265, -122.0114..."
4,5,2024-25,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,81,13.7,412,69.8,1.273929,5.363392e+06,13741.272894,"POLYGON ((-122.29663 37.8311, -122.29778 37.83..."


In [8]:
def spatial_join_school_district(df, districts_gdf, lat_col="Latitude", lon_col="Longitude"):
    hs_districts = districts_gdf[districts_gdf["DistrictType"].isin(["High", "Unified"])][["DistrictName", "geometry"]]

    has_coords = df[lat_col].notna() & df[lon_col].notna()
    points = gpd.GeoDataFrame(
        df.loc[has_coords, [lat_col, lon_col]],
        geometry=gpd.points_from_xy(df.loc[has_coords, lon_col], df.loc[has_coords, lat_col]),
        crs="EPSG:4326",
    )

    joined = gpd.sjoin(points, hs_districts, how="left", predicate="within")
    joined = joined[~joined.index.duplicated(keep="first")]

    result = df.copy()
    result["school_district_spatial"] = "Unknown"
    result.loc[joined.index, "school_district_spatial"] = joined["DistrictName"].fillna("Unknown").values
    return result

train_df = spatial_join_school_district(train_df, districts_gdf)
val_df = spatial_join_school_district(val_df, districts_gdf)
test_df = spatial_join_school_district(test_df, districts_gdf)

match_rate = (train_df["school_district_spatial"] != "Unknown").mean() * 100
print(f"Spatial join matched {match_rate:.1f}% of training rows to a school district.")
train_df["school_district_spatial"].value_counts().head(10)


Spatial join matched 99.9% of training rows to a school district.


school_district_spatial
Los Angeles Unified           36822
San Diego Unified             10196
Antelope Valley Union High     7227
Capistrano Unified             6812
Grossmont Union High           6769
Desert Sands Unified           6766
Perris Union High              6675
Palm Springs Unified           5758
Chaffey Joint Union High       5093
William S. Hart Union High     5082
Name: count, dtype: int64

## Feature 5: ZIP-Level Historical Price-per-Sqft (CV-Safe)

Split the training homes into 5 random chunks. For each chunk, compute the ZIP averages using only the other 4 chunks. Then hand those averages to the chunk that was left out. So no home ever "sees" its own price reflected back at it. Rotate through all 5 chunks so everyone gets covered.

The process:

- **Training rows**: 5-fold out-of-fold assignment: each fold's ZIP
  aggregate is computed only from the *other* four folds, so no row ever
  contributes to its own encoded value.
- **Validation/test rows**: use the aggregate computed from *all* of training
  — no leakage risk there, since val/test rows were never part of the
  aggregate to begin with.
- **Unseen ZIPs**: fall back to the global training median.


In [10]:
def zip_price_per_sqft_lookup(df):
    valid = df["LivingArea"].notna() & (df["LivingArea"] > 0) & df[TARGET_COL].notna()
    price_per_sqft = df.loc[valid, TARGET_COL] / df.loc[valid, "LivingArea"]
    return price_per_sqft.groupby(df.loc[valid, "PostalCode"]).median()


global_median_pps = (train_df[TARGET_COL] / train_df["LivingArea"].replace(0, np.nan)).median()

# --- Training rows: 5-fold out-of-fold assignment ---
train_df = train_df.reset_index(drop=True)
train_df["zip_price_per_sqft"] = np.nan

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
for fold_train_idx, fold_holdout_idx in kf.split(train_df):
    fold_lookup = zip_price_per_sqft_lookup(train_df.iloc[fold_train_idx])
    mapped = train_df.iloc[fold_holdout_idx]["PostalCode"].map(fold_lookup)
    train_df.loc[train_df.index[fold_holdout_idx], "zip_price_per_sqft"] = mapped.values

train_df["zip_price_per_sqft"] = train_df["zip_price_per_sqft"].fillna(global_median_pps)

# --- Validation/test: use the full-train-derived lookup ---
full_train_lookup = zip_price_per_sqft_lookup(train_df)
val_df["zip_price_per_sqft"] = val_df["PostalCode"].map(full_train_lookup).fillna(global_median_pps)
test_df["zip_price_per_sqft"] = test_df["PostalCode"].map(full_train_lookup).fillna(global_median_pps)

print(f"Global fallback (train median $/sqft): {global_median_pps:,.0f}")
train_df["zip_price_per_sqft"].describe()


Global fallback (train median $/sqft): 529


count    382529.000000
mean        585.216040
std         295.316319
min          60.366309
25%         355.186520
50%         536.007412
75%         711.304487
max        2749.599573
Name: zip_price_per_sqft, dtype: float64

## Build Old vs. New Feature Sets

In [11]:
engineered_numeric = ["bed_bath_ratio", "property_age_years", "close_month_sin", "close_month_cos", "zip_price_per_sqft"]
engineered_categorical = ["school_district_spatial"]

new_numeric_features = old_numeric_features + engineered_numeric
new_categorical_features = old_categorical_features + engineered_categorical

def make_xy(df, numeric_features, categorical_features):
    return df[numeric_features + categorical_features].copy(), df[TARGET_COL].copy()

X_train_old, y_train = make_xy(train_df, old_numeric_features, old_categorical_features)
X_val_old, y_val = make_xy(val_df, old_numeric_features, old_categorical_features)
X_test_old, y_test = make_xy(test_df, old_numeric_features, old_categorical_features)

X_train_new, _ = make_xy(train_df, new_numeric_features, new_categorical_features)
X_val_new, _ = make_xy(val_df, new_numeric_features, new_categorical_features)
X_test_new, _ = make_xy(test_df, new_numeric_features, new_categorical_features)

print(f"Old feature count: {X_train_old.shape[1]}   New feature count: {X_train_new.shape[1]}")


Old feature count: 34   New feature count: 40


## Preprocessing & Models

Same architecture choices established in `model_comparison.ipynb`: `Ridge`
(not plain `LinearRegression`, for the numerical-stability reasons diagnosed
there) for the linear model, no scaling for tree models, log-transformed
target throughout, `min_frequency` capping on one-hot encoding to limit
sparse-collinearity from high-cardinality categoricals (now including the new
`school_district_spatial` field).


In [12]:
def make_preprocessors(numeric_features, categorical_features):
    numeric_linear = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    numeric_tree = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ])
    preprocessor_linear = ColumnTransformer([
        ("numeric", numeric_linear, numeric_features), ("categorical", categorical, categorical_features),
    ])
    preprocessor_tree = ColumnTransformer([
        ("numeric", numeric_tree, numeric_features), ("categorical", categorical, categorical_features),
    ])
    return preprocessor_linear, preprocessor_tree


def make_model(regressor):
    return TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)


def make_models(preprocessor_linear, preprocessor_tree):
    return {
        "Linear Regression (Ridge)": Pipeline([
            ("preprocess", preprocessor_linear), ("model", make_model(Ridge(alpha=RIDGE_ALPHA))),
        ]),
        "Decision Tree": Pipeline([
            ("preprocess", preprocessor_tree),
            ("model", make_model(DecisionTreeRegressor(max_depth=20, min_samples_leaf=25, random_state=RANDOM_SEED))),
        ]),
        "Random Forest": Pipeline([
            ("preprocess", preprocessor_tree),
            ("model", make_model(RandomForestRegressor(
                n_estimators=300, max_depth=None, min_samples_leaf=5,
                max_features="sqrt", random_state=RANDOM_SEED, n_jobs=-1,
            ))),
        ]),
    }


## Fit & Evaluate: Old vs. New, Every Model, on Validation


In [13]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ape = np.abs(y_true - y_pred) / y_true
    return {
        "r2": r2_score(y_true, y_pred), "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mape": float(np.mean(ape) * 100), "mdape": float(np.median(ape) * 100),
    }


preprocessor_linear_old, preprocessor_tree_old = make_preprocessors(old_numeric_features, old_categorical_features)
preprocessor_linear_new, preprocessor_tree_new = make_preprocessors(new_numeric_features, new_categorical_features)

models_old = make_models(preprocessor_linear_old, preprocessor_tree_old)
models_new = make_models(preprocessor_linear_new, preprocessor_tree_new)

val_results = []
val_predictions = {}

for feature_set, models, X_tr, X_va in [
    ("Old", models_old, X_train_old, X_val_old),
    ("New", models_new, X_train_new, X_val_new),
]:
    for name, pipe in models.items():
        print(f"Training {name} ({feature_set} features)...")
        pipe.fit(X_tr, y_train)
        pred = pipe.predict(X_va)
        metrics = regression_metrics(y_val, pred)
        metrics.update({"model": name, "feature_set": feature_set})
        val_results.append(metrics)
        val_predictions[(name, feature_set)] = pred

val_results_df = pd.DataFrame(val_results)
val_pivot = val_results_df.pivot(index="model", columns="feature_set", values=["r2", "mae", "rmse", "mape", "mdape"])
val_pivot


Training Linear Regression (Ridge) (Old features)...
Training Decision Tree (Old features)...
Training Random Forest (Old features)...
Training Linear Regression (Ridge) (New features)...
Training Decision Tree (New features)...
Training Random Forest (New features)...


r2                      mae                 \
feature_set                     New       Old            New            Old   
model                                                                         
Decision Tree              0.852538  0.846234  188720.829025  194594.184125   
Linear Regression (Ridge)  0.809014  0.797672  221175.779223  224609.472578   
Random Forest              0.746969  0.697325  230039.939312  259771.800374   

                                    rmse                      mape             \
feature_set                          New            Old        New        Old   
model                                                                           
Decision Tree              377439.374644  385422.337669  13.459492  13.743817   
Linear Regression (Ridge)  429543.751261  442114.015468  16.015856  15.793934   
Random Forest              494416.798379  540747.453438  15.392051  17.623573   

                               mdape             
feature_set                      New        Old  
model                                            
Decision Tree               9.515858   9.589711  
Linear Regression (Ridge)  11.744933  11.706811  
Random Forest              10.737184  12.412255

## Do the New Features Help, or Is It Noise?

Same bootstrap approach as the model comparison in the previous notebook, 
but now the two things being compared are **the same model architecture,
old features vs. new features**, isolating the effect of feature
engineering.


In [14]:
def bootstrap_mdape_diff(y_true, pred_new, pred_old, n_boot=1000, random_state=RANDOM_SEED):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true, dtype=float)
    pred_new, pred_old = np.asarray(pred_new, dtype=float), np.asarray(pred_old, dtype=float)
    n = len(y_true)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        ape_new = np.abs(y_true[idx] - pred_new[idx]) / y_true[idx]
        ape_old = np.abs(y_true[idx] - pred_old[idx]) / y_true[idx]
        diffs[i] = np.median(ape_new) * 100 - np.median(ape_old) * 100
    return diffs


significance_rows = []
for name in models_old.keys():
    diffs = bootstrap_mdape_diff(y_val, val_predictions[(name, "New")], val_predictions[(name, "Old")])
    significance_rows.append({
        "model": name,
        "mean_mdape_diff": diffs.mean(),
        "ci_low": np.percentile(diffs, 2.5),
        "ci_high": np.percentile(diffs, 97.5),
        "pct_new_features_better": float((diffs < 0).mean() * 100),
    })

significance_df = pd.DataFrame(significance_rows).set_index("model")
significance_df


,mean_mdape_diff,ci_low,ci_high,pct_new_features_better
model,,,,
Linear Regression (Ridge),0.035072,-0.144612,0.223895,36.5
Decision Tree,-0.086105,-0.326554,0.154514,76.9
Random Forest,-1.677780,-1.885905,-1.454428,100.0


## Interpreting the Feature Engineering Bootstrap Results

1. **Random Forest is the only model with a statistically confirmed improvement.**
Its MdAPE dropped by a mean of 1.68 points, and critically, the entire 95%
interval [-1.886, -1.454] sits below zero — every one of the 1,000 bootstrap
resamples favored the new features (100.0%). 

2. **Decision Tree and Ridge are both inconclusive, despite looking different at a glance.** This is the important nuance in this table: `pct_new_features_better`
alone would suggest Decision Tree improved too (76.9% of resamples favored
the new features) — but its confidence interval, [-0.327, 0.155], straddles
zero. That means a meaningful share of resamples actually favored the *old*
features, and the true effect could plausibly be zero or even slightly
negative. Ridge shows
essentially no effect either way (mean diff of +0.035, near-zero, interval
spanning both directions)

**Conclusion:** the new engineered features are validated for the model
that ends up mattering: Random Forest is either the current best performer
or a strong second read on the price-band breakdown, so they're worth
keeping in the final feature set even though they don't universally help
every architecture equally.

## Final Test Evaluation 

Whichever `(model, feature_set)` combination has the best validation MdAPE
gets evaluated on test.


In [15]:
best_combo = val_results_df.set_index(["model", "feature_set"])["mdape"].idxmin()
best_model_name, best_feature_set = best_combo
print("Best (model, feature_set) by validation MdAPE:", best_combo)

final_models = models_new if best_feature_set == "New" else models_old
X_test_final = X_test_new if best_feature_set == "New" else X_test_old

test_pred = final_models[best_model_name].predict(X_test_final)
test_metrics = regression_metrics(y_test, test_pred)

print(f"\nTest set ({PRIMARY_SPLIT}) — {best_model_name}, {best_feature_set} features:")
for k, v in test_metrics.items():
    print(f"  {k.upper():6s}: {v:,.3f}")


Best (model, feature_set) by validation MdAPE: ('Decision Tree', 'New')

Test set (2026-06) — Decision Tree, New features:
  R2    : 0.851
  MAE   : 189,697.409
  RMSE  : 379,446.272
  MAPE  : 13.700
  MDAPE : 9.512


## Feature Importance

Since the winning model is tree-based, feature importances give a direct read on
which of the new engineered features actually earned their place.


In [20]:
fitted = final_models[best_model_name]
feature_names = fitted.named_steps["preprocess"].get_feature_names_out()
importances = fitted.named_steps["model"].regressor_.feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(20)
)
importance_df


,feature,importance
26,numeric__zip_price_per_sqft,0.683452
0,numeric__LivingArea,0.271601
3,numeric__LotSizeSquareFeet,0.009274
9,numeric__Longitude,0.005454
8,numeric__Latitude,0.004993
4,numeric__YearBuilt,0.003478
10,numeric__AssociationFee,0.003059
5,numeric__GarageSpaces,0.001857
22,numeric__bed_bath_ratio,0.001695
23,numeric__property_age_years,0.001452


## Save Comparison Artifacts

In [19]:
val_pivot.to_csv(OUTPUT_DIR / "geo_features_validation_comparison.csv")
significance_df.to_csv(OUTPUT_DIR / "geo_features_significance.csv")

with open(OUTPUT_DIR / "geo_features_test_result.json", "w") as f:
    json.dump({
        "best_model": best_model_name,
        "best_feature_set": best_feature_set,
        "test_metrics": test_metrics,
    }, f, indent=2)

print("Saved comparison artifacts to", OUTPUT_DIR)


Saved comparison artifacts to outputs


## Summary & Reflection

  1. **Which features helped, and for which model:** of the three model
  architectures tested, only **Random Forest** showed a statistically confirmed
  improvement from the new engineered features. The feature engineering earned its place for one architecture (Random Forest), not across all of them.

  2. **Which engineered features actually did the work:** the feature importance
  breakdown for the winning model is stark. `zip_price_per_sqft` alone accounts
  for **68.3%** of total importance, and `LivingArea` for another **27.2%** —
  together, those two features explain roughly 95% of what the model relies
  on. Every other feature — including `Latitude`/`Longitude`, `YearBuilt`,
  `bed_bath_ratio`, `property_age_years`, the school-district spatial join, and
  both cyclical month features — contributes less than 1% each. Of the five
  newly engineered features, only `zip_price_per_sqft` mattered in any
  meaningful way; `bed_bath_ratio`, `property_age_years`, `close_month_sin`,
  and `close_month_cos` are all present but essentially unused by the model
  (each under 0.2%).

  3. **A finding worth sitting with, not just reporting:** `zip_price_per_sqft`
  dominating this heavily is a double-edged result. On one hand, it's a strong
  signal exactly where you'd expect one — location is consistently one of the
  best predictors in real estate, and this feature captures it more directly
  than raw coordinates can. On the other hand, a single derived feature
  accounting for over two-thirds of the model's decision-making is worth
  scrutinizing. It suggests the model may be
  leaning on "what did nearby homes sell for" almost as a proxy for the answer
  itself.

  4. **Why the geometry-based and temporal features underperformed:** the
  school-district spatial join and cyclical month encoding were both motivated
  by real patterns found earlier in this project (missingness in
  `HighSchoolDistrict`, seasonal price cycles in exploration), but neither
  shows up as meaningfully important here. Their
  signal may already be substantially captured by existing features
  (`Latitude`/`Longitude` and `MLSAreaMajor` already encode much of what school
  district would add; `zip_price_per_sqft` may already implicitly reflect
  seasonal effects if computed across a period that includes seasonal
  variation).

**Bottom line:** Random Forest with the new feature set has the strongest
improvement result across all notebooks. The improvement is bootstrap-confirmed,
and the dominant driver is the ZIP-level price
aggregate specifically